In [ ]:
pip install pandas

In [ ]:
pip install matplotlib

In [ ]:
pip install seaborn

In [20]:
# =========================================
# BLOCO 1 — IMPORTAÇÃO E CONFIGURAÇÃO
# =========================================
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

# Estilo visual dos gráficos
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)

print("✅ Bibliotecas carregadas com sucesso")

✅ Bibliotecas carregadas com sucesso


In [24]:
# =========================================
# BLOCO 2 — LOAD DATA
# =========================================

df = pd.read_csv('cars-data-analysis/cars-dataset-2025.csv', encoding='latin1')
print("✅ Dataset carregado com sucesso")

# Validação
display(df.head())
print("\nShape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: 'cars-data-analysis/cars-dataset-2025.csv'

In [ ]:
# =========================================
# BLOCO 3 — RENOMEAR COLUNAS
# =========================================

df.columns = [
    'company', 'car_name', 'engine', 'capacity',
    'hp', 'top_speed', 'acceleration_0_100',
    'price', 'fuel', 'seats', 'torque'
]

# Validação
print("✅ Colunas atualizadas:")
print(df.columns)

In [ ]:
# =========================================
# BLOCO 4 — LIMPEZA NUMÉRICA
# =========================================

def extract_number(x):
    """
    Extrai números de strings com unidades (ex: '800 Nm', '100-200').
    Retorna média se houver intervalo.
    """
    if pd.isnull(x):
        return np.nan
    
    x = str(x)
    nums = re.findall(r'\d+', x.replace(',', ''))
    
    if not nums:
        return np.nan
    
    return float(np.mean([float(n) for n in nums]))

cols_to_clean = ['hp', 'top_speed', 'acceleration_0_100', 'torque', 'capacity', 'price', 'seats']

for col in cols_to_clean:
    df[col] = df[col].apply(extract_number)

# Validação
display(df[cols_to_clean].head())

In [ ]:
# =========================================
# BLOCO 4 — LIMPEZA NUMÉRICA
# =========================================

def extract_number(x):
    """
    Extrai números de strings com unidades (ex: '800 Nm', '100-200').
    Retorna média se houver intervalo.
    """
    if pd.isnull(x):
        return np.nan
    
    x = str(x)
    nums = re.findall(r'\d+', x.replace(',', ''))
    
    if not nums:
        return np.nan
    
    return float(np.mean([float(n) for n in nums]))

cols_to_clean = ['hp', 'top_speed', 'acceleration_0_100', 'torque', 'capacity', 'price', 'seats']

for col in cols_to_clean:
    df[col] = df[col].apply(extract_number)

# Validação
display(df[cols_to_clean].head())

In [ ]:
# =========================================
# BLOCO 6 — NORMALIZAÇÃO DE FUEL
# =========================================

def normalize_fuel(text):
    """
    Padroniza texto de combustível:
    - lowercase
    - corrige erros
    - remove separadores
    """
    if pd.isnull(text):
        return None
    
    text = text.lower()
    text = text.replace('hyrbrid', 'hybrid')
    text = re.sub(r'[,/()]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['fuel_norm'] = df['fuel'].apply(normalize_fuel)

# Validação
display(df[['fuel', 'fuel_norm']].head(10))

In [ ]:
# =========================================
# BLOCO 7 — CLASSIFICAÇÃO DE FUEL
# =========================================

def classify_fuel(text):
    """
    Classifica combustível em categorias principais.
    """
    if text is None:
        return 'Other'
    
    tokens = set(text.split())
    
    if 'hydrogen' in tokens:
        return 'Hydrogen'
    
    if 'electric' in tokens or 'ev' in tokens:
        if 'hybrid' in tokens:
            return 'Hybrid'
        return 'Electric'
    
    if 'hybrid' in tokens:
        return 'Hybrid'
    
    if 'cng' in tokens:
        return 'CNG'
    
    if 'diesel' in tokens and 'petrol' in tokens:
        return 'Flex Fuel'
    
    if 'diesel' in tokens:
        return 'Diesel'
    
    if 'petrol' in tokens or 'gas' in tokens:
        return 'Gasoline'
    
    return 'Other'

df['fuel_clean'] = df['fuel_norm'].apply(classify_fuel)

# Validação
print(df['fuel_clean'].value_counts())

In [ ]:
# =========================================
# BLOCO 8 — FEATURE ENGINEERING
# =========================================

# Relação custo-benefício
df['hp_per_price'] = df['hp'] / df['price']

# Categoria de performance
df['performance_category'] = pd.cut(
    df['acceleration_0_100'],
    bins=[0, 4, 7, 12, 20],
    labels=['Supercar', 'Sport', 'Normal', 'Slow']
)

# Validação
display(df[['hp_per_price', 'performance_category']].head())

In [ ]:
# =========================================
# BLOCO 9 — DISTRIBUIÇÃO DE PREÇO
# =========================================

plt.figure()
sns.histplot(df['price'], bins=25, kde=True, color='skyblue')
plt.title('Distribuição de Preços')
plt.xlabel('Preço')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
# =========================================
# BLOCO 10 — HP vs PREÇO
# =========================================

plt.figure()
sns.scatterplot(data=df, x='hp', y='price', hue='fuel_clean', palette='viridis')
plt.title('HP vs Preço')
plt.show()

In [ ]:
# =========================================
# BLOCO 11 — COMBUSTÍVEL
# =========================================

plt.figure()
sns.countplot(data=df, x='fuel_clean', hue='fuel_clean', palette='Set2')
plt.title('Distribuição de Combustível')
plt.xticks(rotation=45)
plt.legend([],[], frameon=False)
plt.show()

In [ ]:
# =========================================
# BLOCO 12 — TOP MARCAS
# =========================================

top_brands = df.groupby('company')['price'].mean().sort_values(ascending=False).head(10)

plt.figure()
sns.barplot(x=top_brands.values, y=top_brands.index, palette='coolwarm')
plt.title('Top 10 Marcas por Preço Médio')
plt.show()

In [ ]:
# =========================================
# BLOCO 13 — HEATMAP
# =========================================

plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Mapa de Correlação')
plt.show()

In [ ]:
# =========================================
# BLOCO 14 — CUSTO BENEFÍCIO
# =========================================

best = df.sort_values(by='hp_per_price', ascending=False).head(10)

plt.figure()
sns.barplot(x=best['hp_per_price'], y=best['car_name'], palette='magma')
plt.title('Top 10 Custo-Benefício')
plt.show()

In [ ]:
# =========================================
# BLOCO 15 — EXPORTAÇÃO FINAL
# =========================================

try:
    df.to_csv('cars_cleaned_final.csv', index=False)
    print("✅ Dataset final salvo com sucesso")
except Exception as e:
    print("❌ Erro ao salvar:", e)